# Module 10: Production Deployment Roadmap

This notebook captures the production implementation plan for Chimera, the dashboard wireframe, and the operating rules that govern retraining and rollback.

The source of truth for reusable logic lives in `src/deployment_plan.py`.

## System architecture blueprint

```mermaid
flowchart LR
    A[Weekly raw data refresh] --> B[Batch layer\nALS, MBA, utility scoring]
    B --> C[Versioned recommendation artifacts]
    C --> D[FastAPI serving layer\nTop-5 per household]
    D --> E[Power BI executive dashboard]
    D --> F[Event logging and clickstream]
    F --> G[Monitoring layer\nprecision, margin, drift]
    G --> B
```

The production design keeps the scoring logic in `src/`, exposes a lightweight serving layer, and uses monitoring to decide when to retrain or roll back.

In [2]:
from pathlib import Path

from IPython.display import display
import sys
import os

# Adds the project root to the sys.path
sys.path.append(os.path.abspath(os.path.join('..')))

from src.deployment_plan import build_deployment_roadmap

artifacts = build_deployment_roadmap()

# Show the production tables that back the Power BI wireframe.
display(artifacts.architecture_table)
display(artifacts.dashboard_wireframe_table)
display(artifacts.retraining_policy_table)
display(artifacts.uniqueness_table)

,Layer,Responsibility,Inputs,Outputs,Suggested stack,Monitoring signal
0,Batch learning,"Refresh ALS factors, MBA rules, utility scores...","Weekly transaction extracts, campaign tables, ...",Versioned recommendation artifacts and validat...,Airflow + pandas + implicit + mlxtend,"Training freshness, data completeness, offline..."
1,Serving API,Return Top-5 recommendations for the latest ho...,"Household key, recent basket items, cached fea...",Ranked recommendation payload with utility dec...,FastAPI + Redis + JSON schema,"Latency, cache hit rate, clickthrough rate"
2,Monitoring,"Track uptake, margin lift, drift, and rollback...","Recommendation logs, clickstream events, reali...",Health dashboards and alert thresholds.,Power BI + Prometheus + Great Expectations,"Precision@5, margin lift, deal_sensitivity drift"
3,Governance,"Version, approve, and roll back releases befor...","Model registry versions, validation reports, A...",Approved release notes and rollback decisions.,MLflow + approval checklist,"Release audit trail, approval status"


,Screen,Audience,Core visuals,Primary metrics,Business question
0,Business Health,Executives,"Trend cards, baseline comparison, active-user ...","Total incremental margin, basket diversity lif...",Is Chimera generating more money than the base...
1,Micro-View,Category managers,"Household profile, recent basket, recommendati...","Archetype, clickthrough rate, top pick, utilit...",What should we show this household right now?
2,Explainability,Merchandising and analytics,Stacked utility bar and natural-language why-t...,"Relevance, uplift, margin, context contributions.",Why did the model choose this item?
3,Model Health,MLOps and analytics,"Precision@5 line, average margin line, drift i...","Precision@5, average recommended margin, deal_...",When do we retrain or roll back?


,Trigger,Threshold,Action,Owner
0,Population behavior shift,deal_sensitivity changes by more than 5% from ...,Schedule a fresh batch run through Airflow.,MLOps lead
1,Offline quality regression,Precision@5 falls by more than 3% on the valid...,"Hold release and review candidate generation, ...",Analytics lead
2,Campaign or promotion drift,current promotion mix differs materially from ...,Refresh campaign flags and promoted commodity ...,Data engineering
3,Health gate failure,"latency, clickthrough, or margin lift violates...",rollback to the previous approved artifact whe...,MLOps lead


,Dimension,Standard recsys,Chimera
0,Objective,Maximize proxy relevance or predicted click pr...,Maximize incremental retailer margin under uti...
1,Scoring logic,"Single ranking score, often opaque to stakehol...","Weighted sum of relevance, uplift, margin, and..."
2,Business control,Recommendation quality is discussed after depl...,"Margin, promotion context, and habit strength ..."
3,Validation,Offline accuracy is the main proof point.,"Incremental Precision@5, ablation, and simulat..."
4,Operations,Model retraining is sometimes ad hoc.,"Weekly retraining, drift thresholds, and rollb..."


In [3]:
from pathlib import Path
import sys

from pprint import pprint

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.deployment_plan import build_deployment_roadmap, export_deployment_roadmap

artifacts = build_deployment_roadmap()
export_paths = export_deployment_roadmap(
    artifacts,
    Path("reports") / "figures",
)

export_paths

{'system_architecture': WindowsPath('reports/figures/system_architecture.html'),
 'dashboard_wireframe': WindowsPath('reports/figures/dashboard_wireframe.pdf'),
 'uniqueness_table': WindowsPath('reports/figures/uniqueness_table.html')}

## Operational rollout and rollback

This roadmap is designed to support weekly batch retraining, API serving, dashboard monitoring, and explicit rollback gates.

Retraining is triggered when population deal sensitivity shifts materially, offline Precision@5 drops, or promotion mix drifts away from the training window.

If health gates fail, the production system rolls back to the previous approved artifact before any wider rollout.